<a href="https://colab.research.google.com/github/bmulaw/NLP-Notebooks/blob/main/NLP_01_traditional_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traditional NLP with NLTK: a three-hour workshop

**Student tutorial and exercises** · Python beginners in natural language processing · English-language data

The workshop alternates guided examples and independent work: complete exercise 1 before starting the guided introduction to exercise 2. The guided sections total one hour; independent and semi-guided exercises total two hours. Timings are indicative.
The instructor's notebook contains complete solutions and can be shared after the workshop.

**Learning goals:** build a vocabulary and an inverted index; compare sets and count vectors; compute TF-IDF; annotate sentiment using a lexicon; use NLTK named entities to retrieve movie reviews and summarize their sentiment.

| Time | Work |
|---|---|
| 0–25 min | Guided text preparation, indexing and Jaccard |
| 25–35 min | Guided TF-IDF |
| 35–85 min | Independent exercise 1: similarity matrices |
| 85–100 min | Guided IMDb, preprocessing, entities and sentiment lexicon |
| 100–140 min | Independent exercise 2: IMDb sentiment and entity search |
| 140–170 min | Semi-guided exercise 3: VADER scores |
| 170–180 min | Guided exercise 4: evaluation on IMDb test data |

**How to run:** Open [Google Colab](https://colab.research.google.com/), choose **File > Upload notebook**, connect a Python runtime, then execute the guided cells in order using Shift+Enter. A Google account and an Internet connection are needed for the first installation and downloads. CPU is enough; no GPU, API key or Hugging Face account is required.

The exercises have function skeletons and checks. In the student notebook, replace each `raise NotImplementedError(...)` with your own code before running the corresponding checks. Run the cells in order after completing them. Your work is not automatically sent or published anywhere.

**Libraries:** NLTK performs the NLP work. Hugging Face `datasets` downloads the original IMDb texts and `evaluate` provides evaluation metrics with scikit-learn as a dependency; Python's standard library handles data structures and arithmetic. No pretrained transformer or machine-learning training is needed.


## 0. Setup and imports

Run the installation once. If you previously imported a different NLTK version in the same runtime, restart the runtime and run the imports again.


In [1]:
%pip -q install "nltk==3.9.2" "datasets==4.3.0" "evaluate==0.4.6" "scikit-learn"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.


In [2]:
import html
import math
import re
import time
from collections import Counter, defaultdict
from importlib.metadata import version

import nltk
from nltk.corpus import stopwords, opinion_lexicon
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tree import Tree
from datasets import load_dataset

print("NLTK:", nltk.__version__)
print("datasets:", version("datasets"))

NLTK: 3.9.2
datasets: 4.3.0


The NLTK package and NLTK data are separate downloads. The English sentence tokenizer, POS tagger and entity chunker need their own data files. This cell checks that every download succeeds.


In [3]:
RESOURCES = [
    "stopwords", "punkt_tab", "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker_tab", "words", "opinion_lexicon", "vader_lexicon",
]
for resource in RESOURCES:
    success = nltk.download(resource, quiet=True)
    if not success:
        raise RuntimeError(f"Could not download NLTK resource: {resource}. Check Internet access and retry.")
print("All NLTK resources are available.")

All NLTK resources are available.


# Part 1 — Comparing short sentences

## Guided 1.1: corpus and preprocessing

A **corpus** is a collection of documents. Here, each sentence is one document. `word_tokenize` splits a string into tokens; lowercasing merges `The` and `the`; `isalpha()` keeps alphabetic tokens; the NLTK stop-word list removes common words. We do not stem or lemmatize in this exercise, so results can be reproduced by hand.

Stop-word removal is a modelling choice. In sentiment analysis, deleting `not` can change the meaning of a sentence, so we will use a different policy in Part 2.


In [4]:
guided_corpus = ["The cat likes milk.", "The dog likes milk."]
stop_words = set(stopwords.words("english"))
print("Original sentences:")
for doc_id, sentence in enumerate(guided_corpus):
    print(f"D{doc_id}: {sentence}")

# Show every preparation step for the first sentence.
original_tokens = word_tokenize(guided_corpus[0], preserve_line=True)
lower_tokens = [token.lower() for token in original_tokens]
alphabetic_tokens = [token for token in lower_tokens if token.isalpha()]
content_tokens = [token for token in alphabetic_tokens if token not in stop_words]
print("Tokens:         ", original_tokens)
print("Lowercase:      ", lower_tokens)
print("Alphabetic only:", alphabetic_tokens)
print("Without stops:  ", content_tokens)

Original sentences:
D0: The cat likes milk.
D1: The dog likes milk.
Tokens:          ['The', 'cat', 'likes', 'milk', '.']
Lowercase:       ['the', 'cat', 'likes', 'milk', '.']
Alphabetic only: ['the', 'cat', 'likes', 'milk']
Without stops:   ['cat', 'likes', 'milk']


In [5]:
def preprocess_for_similarity(text):
    """Return lowercase alphabetic NLTK tokens, excluding English stop words."""
    # preserve_line=True tokenizes this single line without sentence splitting.
    tokens = word_tokenize(text, preserve_line=True)
    normalized = [token.lower() for token in tokens if token.isalpha()]
    return [token for token in normalized if token not in stop_words]

guided_tokens = [preprocess_for_similarity(text) for text in guided_corpus]
for doc_id, tokens in enumerate(guided_tokens):
    print(f"D{doc_id}: {tokens}")
assert guided_tokens == [["cat", "likes", "milk"], ["dog", "likes", "milk"]]

D0: ['cat', 'likes', 'milk']
D1: ['dog', 'likes', 'milk']


## Guided 1.2: vocabulary, forward index and inverted index

A **vocabulary** is the list of distinct tokens across the corpus. We sort it so that each token always has the same column position.
A **forward index** maps a document to its word counts. An **inverted index** maps a word to the document identifiers containing it. A set of document IDs avoids duplicate entries when a word occurs more than once.


In [6]:
# A sorted list gives a reproducible order for vector columns.
vocabulary = sorted({token for tokens in guided_tokens for token in tokens})
# for tokens in guided_tokens:
  # for token in tokens:
      #
token_to_column = {token: column for column, token in enumerate(vocabulary)}
forward_index = {doc_id: Counter(tokens) for doc_id, tokens in enumerate(guided_tokens)}

# defaultdict(set) creates an empty set the first time we see a token.
inverted_index = defaultdict(set)
for doc_id, tokens in enumerate(guided_tokens):
    for token in tokens:
        inverted_index[token].add(doc_id)

print("Vocabulary:", vocabulary)
print("Token to vector column:", token_to_column)
print("Forward index:", {doc_id: dict(counts) for doc_id, counts in forward_index.items()})
print("Inverted index:", {word: sorted(ids) for word, ids in sorted(inverted_index.items())})
print("Documents containing 'milk':", sorted(inverted_index["milk"]))
assert inverted_index["milk"] == {0, 1}

Vocabulary: ['cat', 'dog', 'likes', 'milk']
Token to vector column: {'cat': 0, 'dog': 1, 'likes': 2, 'milk': 3}
Forward index: {0: {'cat': 1, 'likes': 1, 'milk': 1}, 1: {'dog': 1, 'likes': 1, 'milk': 1}}
Inverted index: {'cat': [0], 'dog': [1], 'likes': [0, 1], 'milk': [0, 1]}
Documents containing 'milk': [0, 1]


## Guided 1.3: Jaccard on word bags

Here a bag is converted to a **set** of distinct words. Repetitions do not change Jaccard. With sets A and B:

**J(A, B) = |A ∩ B| / |A ∪ B|.**

Our first sentence contains `{cat, likes, milk}` and the second contains `{dog, likes, milk}`. Their intersection contains two words and their union four, so the result is **2/4 = 0.5**. NLTK exposes Jaccard *distance*, so similarity is `1 - distance`.

Convention for this workshop: two empty sets have similarity 1; an empty and a non-empty set have similarity 0. Libraries can choose another convention.


In [7]:
def jaccard_similarity(words_a, words_b):
    """Return Jaccard similarity of distinct words, including the empty-set convention."""
    set_a, set_b = set(words_a), set(words_b)
    if not set_a and not set_b:
        return 1.0  # Both sentences have the same empty representation.
    return 1.0 - nltk.jaccard_distance(set_a, set_b)

set_a, set_b = set(guided_tokens[0]), set(guided_tokens[1])
print("Intersection:", sorted(set_a & set_b))
print("Union:", sorted(set_a | set_b))
print("Jaccard similarity:", jaccard_similarity(set_a, set_b))
assert jaccard_similarity(set_a, set_b) == 0.5
assert jaccard_similarity([], []) == 1.0
assert jaccard_similarity([], ["cat"]) == 0.0

Intersection: ['likes', 'milk']
Union: ['cat', 'dog', 'likes', 'milk']
Jaccard similarity: 0.5


## Guided 1.4: count vectors and TF-IDF

In a **bag-of-words count vector**, each column is a vocabulary word and its value is the number of occurrences. We first create the same ordered columns for every sentence.

For this workshop, **TF = count of the word / number of retained tokens** and **IDF = log(N / df)**, where N is the number of documents and df the number of documents containing the word. `log` is the natural logarithm. Multiplying TF by IDF gives the weight. Our vocabulary contains only words observed in the corpus, so df is at least 1.

The corpus has two documents. `cat` and `dog` each have df = 1, whereas `likes` and `milk` each have df = 2. Every retained word appears once in a three-token sentence. A word present in every document has IDF = log(2/2) = 0.

**Mini-exercise:** In `tfidf_vectors`, complete the expression after `word:` in the `idf` dictionary. Use `n_documents`, `document_frequency[word]` and `math.log`, then run the cell and check the results.


In [8]:
def count_vector(tokens, vocab):
    """Return word frequencies in the fixed order supplied by vocab."""
    counts = Counter(tokens)
    return [counts[word] for word in vocab]

guided_count_vectors = [count_vector(tokens, vocabulary) for tokens in guided_tokens]
print("Columns:", vocabulary)
for doc_id, vector in enumerate(guided_count_vectors):
    print(f"D{doc_id} count vector: {vector}")
assert guided_count_vectors == [[1, 0, 1, 1], [0, 1, 1, 1]]

Columns: ['cat', 'dog', 'likes', 'milk']
D0 count vector: [1, 0, 1, 1]
D1 count vector: [0, 1, 1, 1]


In [9]:
def tfidf_vectors(tokenized_documents, vocab):
    """Build IDF weights and length-normalized TF-IDF document vectors."""
    n_documents = len(tokenized_documents)
    if n_documents == 0:
        return {}, []
    # Count documents containing a word.
    document_frequency = Counter()
    for tokens in tokenized_documents:
        document_frequency.update(set(tokens))
    idf = {
        word: math.log(n_documents/document_frequency[word])
        for word in vocab
    }
    vectors = []
    for tokens in tokenized_documents:
        counts = Counter(tokens)
        # An empty document is assigned an all-zero vector.
        size = len(tokens)
        vectors.append([(counts[word] / size) * idf[word] if size else 0.0 for word in vocab])
    return idf, vectors

idf, guided_tfidf = tfidf_vectors(guided_tokens, vocabulary)
print("Document frequency:", {word: sum(word in set(doc) for doc in guided_tokens) for word in vocabulary})
print("IDF:", {word: round(weight, 3) for word, weight in idf.items()})
for doc_id, vector in enumerate(guided_tfidf):
    print(f"D{doc_id} TF-IDF:", [round(value, 3) for value in vector])
assert abs(idf["cat"] - (math.log(2))) < 1e-12
assert idf["milk"] == 0.0

Document frequency: {'cat': 1, 'dog': 1, 'likes': 2, 'milk': 2}
IDF: {'cat': 0.693, 'dog': 0.693, 'likes': 0.0, 'milk': 0.0}
D0 TF-IDF: [0.231, 0.0, 0.0, 0.0]
D1 TF-IDF: [0.0, 0.231, 0.0, 0.0]


**Checkpoint:** Why is `cat` weighted more than `milk`? What changes if `cat` occurs twice in D0? On this corpus, a new token absent from both documents is outside the vocabulary. TF-IDF is a document representation; it is not a named-entity tagger or a sentiment classifier.


- **Because cat shows up in less documents than milk. Even if cat occurs twice in D0, it would still have a lower IDF than milk since it shows up in less documents (1 opposed to 2 documents).**

# Independent work: exercise 1

Use the guided functions above. The `assert` checks cover a few corner cases; passing them is a useful start, not a complete proof of correctness. Explain each major step in English comments. Do not read or copy the instructor's solution until after you have tried.

## Exercise 1 — Similarity matrices (50 minutes)

The 10 short invented sentences below are the entire corpus for this exercise. Use the same `preprocess_for_similarity` function as in the tutorial. Identifiers D0–D9 correspond to the order below.


In [10]:
sentences = [
    "The cat chases the mouse.",
    "The dog chases the ball.",
    "The cat likes milk.",
    "The dog likes milk.",
    "The child reads a book.",
    "The child likes the book.",
    "The bird sings in the tree.",
    "The cat sleeps in the tree.",
    "The dog sees the dog.",
    "The child reads a story.",
]
sentence_tokens = [preprocess_for_similarity(text) for text in sentences]
for i, (text, tokens) in enumerate(zip(sentences, sentence_tokens)):
    print(f"D{i}: {text} -> {tokens}")
assert len(sentence_tokens) == 10

D0: The cat chases the mouse. -> ['cat', 'chases', 'mouse']
D1: The dog chases the ball. -> ['dog', 'chases', 'ball']
D2: The cat likes milk. -> ['cat', 'likes', 'milk']
D3: The dog likes milk. -> ['dog', 'likes', 'milk']
D4: The child reads a book. -> ['child', 'reads', 'book']
D5: The child likes the book. -> ['child', 'likes', 'book']
D6: The bird sings in the tree. -> ['bird', 'sings', 'tree']
D7: The cat sleeps in the tree. -> ['cat', 'sleeps', 'tree']
D8: The dog sees the dog. -> ['dog', 'sees', 'dog']
D9: The child reads a story. -> ['child', 'reads', 'story']


### Exercise 1A — Jaccard matrix

1. Create a 10 × 10 list of lists. Entry `[i][j]` is the Jaccard similarity of sentences Di and Dj.
2. Use sets of distinct preprocessed words and the guided `jaccard_similarity` function, including its empty-set convention.
3. Print the matrix with row and column identifiers, rounded for display only. Explain its diagonal and symmetry. Identify the most similar **distinct** pair or pairs; do not compare a sentence to itself.

**Hint:** use two nested loops. D0 and D1 each have three words; only `chases` is shared, so Jaccard(D0, D1) = 1/5 = 0.2.


In [11]:
def jaccard_matrix(documents):
    """Return all pairwise set-Jaccard scores for tokenized documents."""
    # TODO: Build one row per document and one score per document pair.
    jaccard_matrix_pair = []
    for i in range(len(documents)):
      scores = []
      for j in range(len(documents)):
        jaccard_similarity_score = jaccard_similarity(documents[i], documents[j])
        scores.append(jaccard_similarity_score)
      jaccard_matrix_pair.append(scores)
    return jaccard_matrix_pair

In [12]:
def print_matrix(matrix):
    """Display a square similarity matrix with document IDs and two decimal places."""
    print("      " + " ".join(f"D{i:>4}" for i in range(len(matrix))))
    for row_id, row in enumerate(matrix):
        print(f"D{row_id:<3} " + " ".join(f"{value:5.2f}" for value in row))

jaccard_results = jaccard_matrix(sentence_tokens)
assert len(jaccard_results) == 10 and all(len(row) == 10 for row in jaccard_results)
assert abs(jaccard_results[0][1] - 0.2) < 1e-12
assert all(abs(jaccard_results[i][i] - 1.0) < 1e-12 for i in range(10))
assert all(abs(jaccard_results[i][j] - jaccard_results[j][i]) < 1e-12
           for i in range(10) for j in range(10))
print_matrix(jaccard_results)

      D   0 D   1 D   2 D   3 D   4 D   5 D   6 D   7 D   8 D   9
D0    1.00  0.20  0.20  0.00  0.00  0.00  0.00  0.20  0.00  0.00
D1    0.20  1.00  0.00  0.20  0.00  0.00  0.00  0.00  0.25  0.00
D2    0.20  0.00  1.00  0.50  0.00  0.20  0.00  0.20  0.00  0.00
D3    0.00  0.20  0.50  1.00  0.00  0.20  0.00  0.00  0.25  0.00
D4    0.00  0.00  0.00  0.00  1.00  0.50  0.00  0.00  0.00  0.50
D5    0.00  0.00  0.20  0.20  0.50  1.00  0.00  0.00  0.00  0.20
D6    0.00  0.00  0.00  0.00  0.00  0.00  1.00  0.20  0.00  0.00
D7    0.20  0.00  0.20  0.00  0.00  0.00  0.20  1.00  0.00  0.00
D8    0.00  0.25  0.00  0.25  0.00  0.00  0.00  0.00  1.00  0.00
D9    0.00  0.00  0.00  0.00  0.50  0.20  0.00  0.00  0.00  1.00


Use the upper triangle (`i < j`) when searching for the most similar distinct pair, so that each pair is counted once. Include all ties.


In [13]:
def get_max_score(matrix):
  max_res = 0
  for i, scores in enumerate(matrix):
    if (len(scores) == 0 or i == len(matrix)-1): continue
    max_in_scores = max(scores[i+1:])
    max_res = max(max_res, max_in_scores)
  print("max_res", max_res)
  return max_res

def most_similar_pairs(matrix):
    """Return (best score, all index pairs attaining it), excluding the diagonal."""
    # TODO: Look at each distinct pair only once; preserve all tied maxima.
    best_scores = set()
    max_score = get_max_score(matrix)

    for i in range(len(matrix)):
      score = matrix[i]
      for j in range(len(score)):
        if score[j] == max_score and (j,i) not in best_scores:
          best_scores.add((i,j))

    return max_score, best_scores

In [14]:
best_jaccard, pairs_jaccard = most_similar_pairs(jaccard_results)
print("Best Jaccard score:", best_jaccard)
for i, j in pairs_jaccard:
    print(f"D{i} / D{j}: {sentences[i]} | {sentences[j]}")

max_res 0.5
Best Jaccard score: 0.5
D2 / D3: The cat likes milk. | The dog likes milk.
D4 / D9: The child reads a book. | The child reads a story.
D4 / D5: The child reads a book. | The child likes the book.


### Exercise 1B — Cosine matrix from bag-of-words counts

1. Build the vocabulary of all 10 preprocessed sentences in sorted order.
2. Convert each sentence to a **count** vector using the guided `count_vector` function. Repetitions matter here: D8 contains `dog` twice.
3. Use `cosine_distance` from `nltk.cluster.util` in your `cosine_similarity` function, then calculate the 10 × 10 matrix.
4. Print it, find all most-similar distinct pairs, and compare them with the Jaccard results.

**NLTK returns a distance:** `cosine_distance(a, b)` equals `1 - cosine_similarity(a, b)`. For non-zero vectors, subtract this distance from 1 to obtain similarity. **Cosine formula:** `dot(a, b) / (norm(a) * norm(b))`, where the norm is `sqrt(sum(value²))`. Use the same vocabulary order for both vectors. Convention: if either vector is zero, return 0.0 because its mathematical cosine is undefined. A zero vector's diagonal is therefore also 0.0 under this convention.


In [15]:
all_vocabulary = sorted({word for tokens in sentence_tokens for word in tokens})
count_vectors = [count_vector(tokens, all_vocabulary) for tokens in sentence_tokens]
print("Vocabulary:", all_vocabulary)
print("D8 count vector:", count_vectors[8])
print("D8 count for 'dog':", count_vectors[8][all_vocabulary.index("dog")])
assert count_vectors[8][all_vocabulary.index("dog")] == 2

Vocabulary: ['ball', 'bird', 'book', 'cat', 'chases', 'child', 'dog', 'likes', 'milk', 'mouse', 'reads', 'sees', 'sings', 'sleeps', 'story', 'tree']
D8 count vector: [0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0]
D8 count for 'dog': 2


In [16]:
from nltk.cluster.util import cosine_distance

def cosine_similarity(vector_a, vector_b):
    """Return cosine similarity of equal-length count vectors; zero for a zero vector."""
    # TODO: Check dimensions and zero vectors, then use NLTK cosine_distance.
    if len(vector_a) != len(vector_b):
      raise ValueError("Both vectors must be the same size.")
    if not any(vector_a) or not any(vector_b):
      return 0.0
    res = 1.0 - float(nltk.cluster.util.cosine_distance(vector_a, vector_b))
    # print(res)
    return res

def cosine_matrix(vectors):
    # return [[cosine_similarity(i, j) for i in vectors] for j in i]
    """Return cosine similarities for every pair of vectors."""
    # TODO: Use two loops and cosine_similarity.
    res = []
    for i in range(len(vectors)):
      running = []
      for j in range(len(vectors)):
        cos_sim_score = cosine_similarity(vectors[i], vectors[j])
        running.append(round(cos_sim_score,3))
      res.append(running)
    # print(res)

    return res

In [17]:
assert abs(cosine_similarity([1, 1], [1, 0]) - 1 / math.sqrt(2)) < 1e-12
assert cosine_similarity([0, 0], [1, 0]) == 0.0
try:
    cosine_similarity([1], [1, 0])
    raise AssertionError("Mismatched dimensions must raise ValueError.")
except ValueError:
    pass
cosine_results = cosine_matrix(count_vectors)
assert len(cosine_results) == 10 and all(len(row) == 10 for row in cosine_results)
assert all(abs(cosine_results[i][i] - 1.0) < 1e-12 for i in range(10))
assert all(abs(cosine_results[i][j] - cosine_results[j][i]) < 1e-12
           for i in range(10) for j in range(10))
print_matrix(cosine_results)
best_cosine, pairs_cosine = most_similar_pairs(cosine_results)
print("Best cosine score:", best_cosine, "pairs:", pairs_cosine)
print("D1 / D8 (Jaccard, cosine):", jaccard_results[1][8], cosine_results[1][8])

      D   0 D   1 D   2 D   3 D   4 D   5 D   6 D   7 D   8 D   9
D0    1.00  0.33  0.33  0.00  0.00  0.00  0.00  0.33  0.00  0.00
D1    0.33  1.00  0.00  0.33  0.00  0.00  0.00  0.00  0.52  0.00
D2    0.33  0.00  1.00  0.67  0.00  0.33  0.00  0.33  0.00  0.00
D3    0.00  0.33  0.67  1.00  0.00  0.33  0.00  0.00  0.52  0.00
D4    0.00  0.00  0.00  0.00  1.00  0.67  0.00  0.00  0.00  0.67
D5    0.00  0.00  0.33  0.33  0.67  1.00  0.00  0.00  0.00  0.33
D6    0.00  0.00  0.00  0.00  0.00  0.00  1.00  0.33  0.00  0.00
D7    0.33  0.00  0.33  0.00  0.00  0.00  0.33  1.00  0.00  0.00
D8    0.00  0.52  0.00  0.52  0.00  0.00  0.00  0.00  1.00  0.00
D9    0.00  0.00  0.00  0.00  0.67  0.33  0.00  0.00  0.00  1.00
max_res 0.667
Best cosine score: 0.667 pairs: {(2, 3), (4, 9), (4, 5)}
D1 / D8 (Jaccard, cosine): 0.25 0.516


**Discussion:** D8 repeats `dog`. Explain why count cosine can change when a word is repeated while set Jaccard does not. Both methods use shared surface words; neither automatically knows that `cat` and `dog` are related. Describe one advantage and one limitation of each measure.

- **Well cosine tracks the frequency of a word via bag of words but on the contrary, Jaccard counts the existence of a word once (regardless of its frequency). They're both pretty surface level because they don't actually embed any meaning to the word, but instead build a vector based on existence of words. Cosine gives us insight to words that show up most frequently (which could be important) but some words may not be relevant (like the word "like"). Jaccard gives us a valuable comparative tool between texts, provided they have somewhat similar words (irregardless of count of each word), but the limit is that although two texts may have the same word that word could mean something completely different in the two context (lacks attention?).**


### Exercise 1C — Cosine matrix with TF-IDF weights

Use the same 10 preprocessed sentences and vocabulary as in exercises 1A and 1B.

1. Reuse `tfidf_vectors(sentence_tokens, all_vocabulary)` to calculate TF-IDF weights on the **10-sentence corpus**. Complete the earlier guided IDF mini-exercise first.
2. Apply `cosine_matrix` from exercise 1B to the resulting vectors and display the 10 × 10 similarity matrix.
3. Find all the most similar distinct pairs with `most_similar_pairs`. Compare the results with the cosine matrix based on word counts.
4. Explain how IDF affects words appearing in many documents. What happens if a document has an all-zero TF-IDF vector?

Use the same convention as before: cosine similarity is 0.0 if either vector is zero. IDF is `log(N / df)`; compute it from these 10 sentences, not from the two-sentence tutorial corpus.


In [40]:
def tfidf_cosine_matrix(documents, vocab):
    """Return the pairwise cosine matrix of TF-IDF document vectors."""
    # TODO: Compute TF-IDF vectors with tfidf_vectors.
    # TODO: Return their pairwise similarities using cosine_matrix.
    _, tfidf_vectors = tfidf_vectors(documents, vocab)
    return cosine_matrix(tfidf_vectors)

In [41]:
tfidf_results = tfidf_cosine_matrix(sentence_tokens, all_vocabulary)
assert len(tfidf_results) == 10 and all(len(row) == 10 for row in tfidf_results)
assert all(abs(tfidf_results[i][j] - tfidf_results[j][i]) < 1e-12
           for i in range(10) for j in range(10))

# These ten sentences all have at least one non-zero TF-IDF weight.
assert all(abs(tfidf_results[i][i] - 1.0) < 1e-12 for i in range(10))
print_matrix(tfidf_results)
best_tfidf, pairs_tfidf = most_similar_pairs(tfidf_results)
print("Best TF-IDF cosine score:", best_tfidf, "pairs:", pairs_tfidf)
print("D1 / D8 — count cosine:", cosine_results[1][8],
      "TF-IDF cosine:", tfidf_results[1][8])

      D   0 D   1 D   2 D   3 D   4 D   5 D   6 D   7 D   8 D   9
D0    1.00  0.28  0.20  0.00  0.00  0.00  0.00  0.15  0.00  0.00
D1    0.28  1.00  0.00  0.20  0.00  0.00  0.00  0.00  0.28  0.00
D2    0.20  0.00  1.00  0.74  0.00  0.26  0.00  0.20  0.00  0.00
D3    0.00  0.20  0.74  1.00  0.00  0.26  0.00  0.00  0.37  0.00
D4    0.00  0.00  0.00  0.00  1.00  0.67  0.00  0.00  0.00  0.51
D5    0.00  0.00  0.26  0.26  0.67  1.00  0.00  0.00  0.00  0.20
D6    0.00  0.00  0.00  0.00  0.00  0.00  1.00  0.23  0.00  0.00
D7    0.15  0.00  0.20  0.00  0.00  0.00  0.23  1.00  0.00  0.00
D8    0.00  0.28  0.00  0.37  0.00  0.00  0.00  0.00  1.00  0.00
D9    0.00  0.00  0.00  0.00  0.51  0.20  0.00  0.00  0.00  1.00
max_res 0.736
Best TF-IDF cosine score: 0.736 pairs: {(2, 3)}
D1 / D8 — count cosine: 0.516 TF-IDF cosine: 0.285


# Part 2 — IMDb movie reviews

## Guided 2.1: load the original corpus

We use the **25,000 labelled reviews in the official IMDb training split** as our working corpus. Every review in this split will receive a lexicon-based label during independent work. The original IMDb dataset also has a separate 25,000-review test split and an unsupervised split; they are outside this workshop's working corpus.

`datasets` downloads the original English texts and the existing labels: 0 = negative, 1 = positive. The labels are reference labels for an optional evaluation, **not** the rule used to predict sentiment. The first download can take several minutes. Review text preserves uppercase letters, which is important for NLTK's named-entity recognizer. The separate NLTK `movie_reviews` corpus was distributed in lowercase, so it is less suitable for this part.


In [20]:
started = time.perf_counter()
imdb = load_dataset("stanfordnlp/imdb", split="train")
print("Number of labelled reviews in the working corpus:", len(imdb))
print("Fields:", imdb.column_names)
print("Reference label names:", imdb.features["label"].names)
print("First original review (first 500 characters):")
print(imdb[0]["text"][:500])
print(f"Dataset loading: {time.perf_counter() - started:.1f} s")
assert len(imdb) == 25_000

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Number of labelled reviews in the working corpus: 25000
Fields: ['text', 'label']
Reference label names: ['neg', 'pos']
First original review (first 500 characters):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent
Dataset loading: 9.6 s


## Guided 2.2: keep two versions of the text

We remove HTML tags and normalize whitespace, but **keep case and punctuation** for named-entity recognition (NER). For lexicon matching, we also prepare lowercase alphabetic tokens. This simple sentiment policy keeps `not`, `no` and `nor` if English stop words are removed, but it does **not** model negation: `not good` still contains the positive word `good`. Contractions such as `n't` are not handled by our alphabetic filter.

Entity recognition receives full original sentences. Never feed it the lowercase, punctuation-free tokens used for word counting: capitalization and word order are useful signals.


In [21]:
def clean_review(text):
    """Remove simple IMDb HTML tags and normalize whitespace while preserving case."""
    decoded = html.unescape(text)
    without_tags = re.sub(r"<[^>]+>", " ", decoded)
    return re.sub(r"\s+", " ", without_tags).strip()

# Retain negation words when removing ordinary stop words.
sentiment_stop_words = stop_words - {"not", "no", "nor"}

def sentiment_tokens(text):
    """Return lowercase alphabetic tokens for lexicon matching; keep negation words."""
    # word_tokenize is an NLTK tokenizer; preserve_line avoids redundant sentence splitting.
    tokens = word_tokenize(clean_review(text), preserve_line=True)
    return [token.lower() for token in tokens
            if token.isalpha() and token.lower() not in sentiment_stop_words]

review_text = clean_review(imdb[0]["text"])
print("Original cleaned text:", review_text[:300])
print("NLTK sentiment tokens:", sentiment_tokens(review_text)[:30])
print("Negation example:", sentiment_tokens("The movie was not good."))
assert "not" in sentiment_tokens("The movie was not good.")

Original cleaned text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h
NLTK sentiment tokens: ['rented', 'video', 'store', 'controversy', 'surrounded', 'first', 'released', 'also', 'heard', 'first', 'seized', 'customs', 'ever', 'tried', 'enter', 'country', 'therefore', 'fan', 'films', 'considered', 'controversial', 'really', 'see', 'plot', 'centered', 'around', 'young', 'swedish', 'drama', 'student']
Negation example: ['movie', 'not', 'good']


## Guided 2.3: named-entity recognition with NLTK

NLTK's classical recognizer expects **POS-tagged tokens**. We first split the text into sentences (`sent_tokenize`), tokenize each sentence (`word_tokenize`), label grammatical categories (`pos_tag`), then build an entity tree using the NLTK maximum-entropy chunker. A named entity is a subtree labelled `PERSON`, `ORGANIZATION`, `GPE`, etc.; its leaves contain `(word, POS-tag)` pairs.

The first example is invented so that its original capitalization is visible. The second is a real IMDb review. Predictions are automatic and can be incomplete or wrong; titles and people are not guaranteed to be found correctly.


In [22]:
# Load the classical NLTK chunker once, then reuse it for every sentence.
entity_chunker = nltk.chunk.ne_chunker(fmt="multiclass")

def entities_in_sentence(sentence):
    """Return (entity text, NLTK entity type) pairs from one original-case sentence."""
    tokens = word_tokenize(sentence, preserve_line=True)
    if not tokens:
        return []
    tagged_tokens = nltk.pos_tag(tokens)
    entity_tree = entity_chunker.parse(tagged_tokens)
    entities = []
    for node in entity_tree:
        # Tree nodes represent named entities; plain tuples represent other words.
        if isinstance(node, Tree):
            name = " ".join(word for word, pos in node.leaves())
            entities.append((name, node.label()))
    return entities

demo_sentence = "Tom Hanks and Meryl Streep visited New York."
demo_tokens = word_tokenize(demo_sentence, preserve_line=True)
demo_tags = nltk.pos_tag(demo_tokens)
demo_tree = entity_chunker.parse(demo_tags)
print("Tokens:", demo_tokens)
print("POS tags:", demo_tags)
print("Entity tree:", demo_tree)
print("Extracted entities:", entities_in_sentence(demo_sentence))

# Display results for only the first three sentences, to keep the guided output short.
for sentence in sent_tokenize(review_text)[:3]:
    print("\nSentence:", sentence[:250])
    print("Entities:", entities_in_sentence(sentence))

Tokens: ['Tom', 'Hanks', 'and', 'Meryl', 'Streep', 'visited', 'New', 'York', '.']
POS tags: [('Tom', 'NNP'), ('Hanks', 'NNP'), ('and', 'CC'), ('Meryl', 'NNP'), ('Streep', 'NNP'), ('visited', 'VBD'), ('New', 'NNP'), ('York', 'NNP'), ('.', '.')]
Entity tree: (S
  (PERSON Tom/NNP)
  (ORGANIZATION Hanks/NNP)
  and/CC
  (PERSON Meryl/NNP Streep/NNP)
  visited/VBD
  (GPE New/NNP York/NNP)
  ./.)
Extracted entities: [('Tom', 'PERSON'), ('Hanks', 'ORGANIZATION'), ('Meryl Streep', 'PERSON'), ('New York', 'GPE')]

Sentence: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967.
Entities: []

Sentence: I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.
Entities: [('U.S.', 'GPE')]

Sentence: The plot is centered around a young Swedish drama student named Lena who wants to learn ev

## Guided 2.4: a dictionary of positive and negative words

NLTK provides the Hu–Liu opinion lexicon as two word lists. The simple rule below counts **occurrences**: a word repeated three times contributes three. Words outside the lists contribute zero. We do not use the IMDb reference label to choose a prediction.

Examples are rules, not human judgments. This lexicon can miss movie-specific uses and cannot reliably handle negation, sarcasm or changes in sentiment within a review. The lexicon is case-insensitive here because we lowercase its entries and our sentiment tokens.


In [23]:
positive_words = set(word.lower() for word in opinion_lexicon.positive())
negative_words = set(word.lower() for word in opinion_lexicon.negative())
print("Positive entries:", len(positive_words), "Negative entries:", len(negative_words))
print("good:", "good" in positive_words, "bad:", "bad" in negative_words)
print("Overlapping entries:", len(positive_words & negative_words))

def lexicon_evidence(text):
    """Count positive and negative token occurrences, and return the matched words."""
    tokens = sentiment_tokens(text)
    positive_matches = [word for word in tokens if word in positive_words]
    negative_matches = [word for word in tokens if word in negative_words]
    return {
        "positive_count": len(positive_matches),
        "negative_count": len(negative_matches),
        "positive_matches": positive_matches,
        "negative_matches": negative_matches,
    }

for text in ["A good, good film.", "A bad and boring film.", "The film is not good."]:
    print(text, "=>", lexicon_evidence(text))
assert lexicon_evidence("good good bad")["positive_count"] == 2
assert lexicon_evidence("good good bad")["negative_count"] == 1

Positive entries: 2006 Negative entries: 4783
good: True bad: True
Overlapping entries: 3
A good, good film. => {'positive_count': 2, 'negative_count': 0, 'positive_matches': ['good', 'good'], 'negative_matches': []}
A bad and boring film. => {'positive_count': 0, 'negative_count': 2, 'positive_matches': [], 'negative_matches': ['bad', 'boring']}
The film is not good. => {'positive_count': 1, 'negative_count': 0, 'positive_matches': ['good'], 'negative_matches': []}


## Independent exercise 2 — Sentiment and named entities in IMDb (40 minutes)

Work on **all 25,000 reviews of the loaded training split** for sentiment classification. For named-entity retrieval we first find candidate sentences that contain the requested text, then run NLTK POS tagging and NER on those sentences. This avoids spending most of the workshop running NER on unrelated reviews. It still checks every review's text for the query and verifies candidate mentions with NLTK. No NER process is run over the unsupervised or test splits.


### Exercise 2A — Binary sentiment from the lexicon

Implement a binary rule for every review:

- Count positive and negative word **occurrences** with `lexicon_evidence`.
- Predict `positive` if the positive count is strictly larger; otherwise predict `negative`.
- Keep the two counts and a Boolean `is_tie` so that ties and reviews with no dictionary evidence remain visible. Sending ties to `negative` is an explicit arbitrary choice made to satisfy the binary output requirement; discuss its consequences.
- Return a record for each review with its dataset row ID, predicted label, counts and tie indicator. Do not use the reference label to make predictions.

Run the small checks first, then run the rule on the entire working corpus. Record runtime, number of labels and number of ties. Optionally compare predictions with IMDb reference labels **afterwards**.


In [24]:
def classify_review(text):
    """Classify one review using positive/negative lexicon occurrence counts."""
    # TODO: Use lexicon_evidence and implement the explicit binary tie rule.
    # Return a dictionary with label, positive_count, negative_count and is_tie.
    lex_evidence = lexicon_evidence(text)
    label = "positive" if lex_evidence["positive_count"] > lex_evidence["negative_count"] else "negative"
    is_tie = lex_evidence["positive_count"] == lex_evidence["negative_count"]
    return {
        "label": label,
        "positive_count": lex_evidence["positive_count"],
        "negative_count": lex_evidence["negative_count"],
        "is_tie": is_tie
    }

def classify_corpus(dataset):
    """Classify every dataset row and return records in dataset order."""
    # TODO: For every row, call classify_review on row["text"] and add row_id.
    # The ground-truth row["label"] must not be used to make predictions.
    count = 0
    res = []
    for i, row in enumerate(dataset):
      lab = classify_review(row["text"])
      lab["row_id"] = i
      # print(lab)
      res.append(lab)
    return res

In [25]:
assert classify_review("good good bad")["label"] == "positive"
assert classify_review("bad bad good")["label"] == "negative"
assert classify_review("good bad")["is_tie"] is True
assert classify_review("good bad")["label"] == "negative"
assert classify_review("table chair")["is_tie"] is True
miniature = [{"text": "good", "label": 0}, {"text": "bad", "label": 1}]
assert [r["label"] for r in classify_corpus(miniature)] == ["positive", "negative"]

In [26]:
started = time.perf_counter()
sentiment_records = classify_corpus(imdb)
sentiment_seconds = time.perf_counter() - started
assert len(sentiment_records) == len(imdb) == 25_000
assert all(record["row_id"] == i for i, record in enumerate(sentiment_records))
print(f"Classified {len(sentiment_records):,} reviews in {sentiment_seconds:.1f} s")
print("Predicted labels:", Counter(record["label"] for record in sentiment_records))
print("Ties:", sum(record["is_tie"] for record in sentiment_records))
print("No lexicon evidence:", sum(record["positive_count"] + record["negative_count"] == 0
                                  for record in sentiment_records))
print("First record:", sentiment_records[0])

Classified 25,000 reviews in 31.2 s
Predicted labels: Counter({'positive': 12995, 'negative': 12005})
Ties: 1838
No lexicon evidence: 19
First record: {'label': 'negative', 'positive_count': 4, 'negative_count': 10, 'is_tie': False, 'row_id': 0}


In [27]:
# Evaluation is separate: only this cell reads the IMDb reference labels.
reference_names = {0: "negative", 1: "positive"}
correct = sum(record["label"] == reference_names[imdb[i]["label"]]
              for i, record in enumerate(sentiment_records))
print(f"Agreement with reference labels on this working split: {correct / len(imdb):.3f}")
print("This is an observation on the working corpus, not an independent test benchmark.")

Agreement with reference labels on this working split: 0.717
This is an observation on the working corpus, not an independent test benchmark.



### Exercise 2B — Retrieve reviews for a named entity

The student chooses an entity name such as `Tom Hanks`. A returned review must satisfy both conditions:

1. The requested name occurs as a case-insensitive **whole phrase** in at least one sentence (so `Tom Hanks` does not match `Tom Hankson`).
2. On at least one such sentence, the NLTK recognizer returns an entity whose complete text matches the query after case-folding and whitespace normalization.

Scan every review for the phrase. Run `entities_in_sentence` **only on candidate sentences**, then use the predicted document label from exercise 2A. Return all matched dataset row IDs and labels. Keep each review once even if the entity occurs several times. Print the number of results by label and display up to two examples in each category (text may be truncated **for display only**). If there are no validated matches, say so; never invent results.

NER is imperfect: a text can contain the name but be missed by NLTK, and aliases such as `Hanks` are not automatically resolved. These restrictions make the retrieval reproducible and the NER requirement explicit.


In [28]:
def normalize_entity(name):
    """Compare entity names ignoring case and repeated whitespace."""
    return " ".join(name.casefold().split())

def entity_pattern(name):
    """Compile a case-insensitive whole-phrase regex, accepting flexible whitespace."""
    words = name.split()
    if not words:
        raise ValueError("Enter a non-empty entity name.")
    # Escape user text so punctuation is treated literally, not as regex syntax.
    phrase = r"\s+".join(re.escape(word) for word in words)
    return re.compile(r"(?<!\w)" + phrase + r"(?!\w)", flags=re.IGNORECASE)
assert entity_pattern("Tom Hanks").search("Tom Hanks visited London.")
assert entity_pattern("Tom Hanks").search("tom   hanks")
assert not entity_pattern("Tom Hanks").search("Tom Hankson")

In [29]:
def find_entity_reviews(dataset, records, entity_name):
    """Return one result per review whose candidate sentence has an NLTK entity match."""
    # TODO: Validate name and matching lengths. Compile entity_pattern(entity_name).
    # TODO: Scan cleaned reviews. Skip reviews without the whole phrase.
    # TODO: Split candidate reviews with sent_tokenize and scan candidate sentences.
    # TODO: Run entities_in_sentence only for a sentence containing the whole phrase.
    # TODO: Confirm normalize_entity(detected name) == normalize_entity(entity_name).
    # TODO: Add one record with row_id, label, entity_type and matching_sentence.

    entity_name_patterned = entity_pattern(entity_name)
    results = []
    for i, review in enumerate(dataset):
      if entity_name_patterned.search(review["text"]):
        sentences = sent_tokenize(review["text"])
        for sentence in sentences:
          if entity_name_patterned.search(sentence):
            entities_type = entities_in_sentence(token)
            results.append({
              "row_id": i,
              "label": records[i]["label"],
              "entity_type": entities_type,
              "matching_sentence": sentence
            })
            break
    print(results)
    return results

These deterministic checks replace the NER function temporarily with a controlled fake. They test filtering, case normalization, duplicate handling and exact boundaries independently of the model's variable predictions. The original NLTK function is restored immediately afterward.


In [30]:
real_entities_in_sentence = entities_in_sentence
try:
    def fake_entities_in_sentence(sentence):
        # Deliberately return the same entity twice to test one result per review.
        return [("Tom Hanks", "PERSON"), ("Tom Hanks", "PERSON")]

    entities_in_sentence = fake_entities_in_sentence
    test_dataset = [
        {"text": "Tom Hanks is good. Tom Hanks is great."},
        {"text": "Tom Hankson is bad."},
        {"text": "No person appears here."},
    ]
    test_records = [{"label": "positive"}, {"label": "negative"}, {"label": "negative"}]
    test_hits = find_entity_reviews(test_dataset, test_records, "tom hanks")
    assert len(test_hits) == 1 and test_hits[0]["row_id"] == 0
    assert test_hits[0]["label"] == "positive"
    assert find_entity_reviews(test_dataset, test_records, "No Such Name") == []
finally:
    entities_in_sentence = real_entities_in_sentence
print("Retrieval checks passed. NLTK NER restored.")

[{'row_id': 0, 'label': 'positive', 'entity_type': [('Tom Hanks', 'PERSON'), ('Tom Hanks', 'PERSON')], 'matching_sentence': 'Tom Hanks is good.'}]
[]
Retrieval checks passed. NLTK NER restored.


In [31]:
ENTITY = "Paris"  # Change this to a full entity name of your choice.
started = time.perf_counter()
entity_results = find_entity_reviews(imdb, sentiment_records, ENTITY)
print(f"NER-confirmed reviews for {ENTITY!r}: {len(entity_results)}")
print("Predicted label counts:", Counter(result["label"] for result in entity_results))
print(f"Search time: {time.perf_counter() - started:.1f} s")
for label in ("positive", "negative"):
    examples = [result for result in entity_results if result["label"] == label]
    print(f"\n{label.upper()} examples: {len(examples)} total")
    for result in examples[:2]:
        print("Row:", result["row_id"], "NER type:", result["entity_type"])
        print("Matching sentence:", result["matching_sentence"])
        print("Review excerpt:", clean_review(imdb[result["row_id"]]["text"])[:400])
if not entity_results:
    print("No NER-confirmed match. Try another complete entity name.")

[{'row_id': 180, 'label': 'negative', 'entity_type': [], 'matching_sentence': "If a man can get away with living in the flat of and professing to love one woman, sleeping (mostly in this very flat) with another, and running around Paris proposing marriage to yet a third, well, I suppose I can wish him the best of luck in the dog-eat-dog world he's chosen to create for himself."}, {'row_id': 461, 'label': 'negative', 'entity_type': [], 'matching_sentence': "Although far, far more intelligent than, say, Paris Hilton (I know, not too difficult) Zizek's mouth spews just as much baloney as hers, just a different kind."}, {'row_id': 507, 'label': 'negative', 'entity_type': [], 'matching_sentence': "I can't stand most reality shows and this one is worst than the one with Paris Hilton, and sure it's his company."}, {'row_id': 765, 'label': 'positive', 'entity_type': [], 'matching_sentence': "French directors and producers, if you don't have the money, the ability or the technology to adapt cor

### Exercise 2C — Entity sentiment score

For the N **NER-confirmed reviews** returned above, let P be the number classified positive and M the number classified negative. Define:

- positivity = P / N
- negativity = M / N
- net score = (P − M) / N, between −1 and +1

These are **document-level lexicon predictions among matched reviews**, not a direct measurement of sentiment expressed *towards* the entity. A review can mention a person while criticizing another aspect of the movie. Ties were assigned to negative by our binary rule. If N = 0, return `None` for all three ratios; do not divide by zero.

Implement the calculation. Display the number of reviews, counts and three scores. Then answer: how would you improve the treatment of negation, entity aliases and sentiment directed at the entity?


In [42]:
def entity_sentiment_summary(matches):
    """Summarize predicted review labels, returning None ratios for no matches."""
    # TODO: Count positive and negative labels and compute the three ratios.
    # Return total, positive, negative, positivity, negativity and net_score.
    total = len(matches)
    if total == 0:
        return {
            "total": total,
            "positive": 0,
            "negative": 0,
            "positivity": None,
            "negativity": None,
            "net_score": None
        }

    pos_count = sum(1 for m in matches if m["label"] == "positive")
    neg_count = sum(1 for m in matches if m["label"] == "negative")
    net_score = (pos_count - neg_count) / total

    return {
        "total": total,
        "positive": pos_count,
        "negative": neg_count,
        "positivity": pos_count/total,
        "negativity": neg_count/total,
        "net_score": net_score
    }

In [43]:
empty = entity_sentiment_summary([])
assert empty["total"] == 0 and empty["net_score"] is None
example = entity_sentiment_summary([
    {"label": "positive"}, {"label": "positive"}, {"label": "negative"}
])
assert example["total"] == 3 and example["positive"] == 2 and example["negative"] == 1
assert abs(example["net_score"] - 1 / 3) < 1e-12
summary = entity_sentiment_summary(entity_results)
print("Entity:", ENTITY)
print("Review-level sentiment summary:", summary)

Entity: Paris
Review-level sentiment summary: {'total': 237, 'positive': 157, 'negative': 80, 'positivity': 0.6624472573839663, 'negativity': 0.33755274261603374, 'net_score': 0.32489451476793246}


# Semi-guided exercise 3 — Opinion scores with VADER (30 minutes)

## Data and objective

Continue with the **25,000 English movie reviews in the IMDb training split** already loaded as `imdb`. Each example has a `text` and a reference `label` (0 negative, 1 positive). Do not use the reference label when calculating a sentiment score.

NLTK's **VADER** uses a lexicon of words and expressions with graded sentiment scores. It also applies rules involving negation, intensifiers, capitalization and punctuation. Its lexicon was designed for short social-media text: performance on movie reviews is an empirical question. The raw lexicon assigns entries scores roughly between −4 and +4; these are word-level scores, not probabilities.

`SentimentIntensityAnalyzer.polarity_scores(text)` combines the lexicon and rules to return:

- `compound`: normalized overall sentiment between −1 and +1;
- `pos`, `neg`, `neu`: normalized positive, negative and neutral indicators. These are not probabilities that the review has a given label.

Keep the original word order, case, punctuation and negations. Use `clean_review` to remove IMDb HTML tags, but do not pass the stop-word-filtered tokens from exercise 2A into VADER.

For this binary exercise, predict `positive` when `compound > 0`, and `negative` otherwise. This assigns a score of exactly 0 to the negative class. It is a deliberate classroom convention, not VADER's only possible threshold. VADER's commonly used three-way setup can instead include a neutral category.

First inspect the example, then complete the two functions below. Compute the scores for every review in the working corpus, compare a few results with the earlier word-count rule, and calculate the mean VADER score for the reviews of your selected entity. The entity score is an average of **whole-review** sentiments, not a targeted analysis of the person or organization.


In [32]:
from nltk.sentiment import SentimentIntensityAnalyzer

# The VADER resource was downloaded in the setup cell.
vader = SentimentIntensityAnalyzer()
print("Lexicon entries:", len(vader.lexicon))
for word in ("good", "excellent", "terrible"):
    print(word, "lexicon score:", vader.lexicon.get(word))

for text in ("The movie was good.", "The movie was not good.", "The movie was excellent!"):
    print(text, "=>", vader.polarity_scores(text))

Lexicon entries: 7502
good lexicon score: 1.9
excellent lexicon score: 2.7
terrible lexicon score: -2.1
The movie was good. => {'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}
The movie was not good. => {'neg': 0.376, 'neu': 0.624, 'pos': 0.0, 'compound': -0.3412}
The movie was excellent! => {'neg': 0.0, 'neu': 0.429, 'pos': 0.571, 'compound': 0.6114}


## Complete the functions

Fill in both functions, then run the checks and the corpus analysis cells in order.

In [44]:
def vader_review(text):
    """Return VADER indicators and a binary label for one review."""
    # TODO: Clean the review with clean_review while retaining case and punctuation.
    # TODO: Call vader.polarity_scores on the cleaned text.
    # TODO: Return a dictionary with compound, pos, neg, neu and label.
    # Use positive if compound > 0, otherwise negative.
    cleaned = clean_review(text)
    scores = vader.polarity_scores(cleaned)
    return {
        "label": "positive" if scores["compound"] > 0 else "negative",
        "compound": scores["compound"],
        "pos": scores["pos"],
        "neg": scores["neg"],
        "neu": scores["neu"]
    }

def vader_corpus(dataset):
    """Score every review and keep its row ID."""
    # TODO: Loop over the dataset, call vader_review on row["text"],
    # add row_id to each result and return the list in dataset order.
    records = []
    for i, row in enumerate(dataset):
        record = vader_review(row["text"])
        record["row_id"] = i
        records.append(record)
    return records

In [45]:
assert vader_review("The movie was good.")["compound"] > 0
assert vader_review("The movie was not good.")["compound"] < 0
assert vader_review("The movie.")["compound"] == 0
assert vader_review("The movie.")["label"] == "negative"
miniature_vader = [{"text": "The movie was good.", "label": 0},
                   {"text": "The movie was not good.", "label": 1}]
assert [r["label"] for r in vader_corpus(miniature_vader)] == ["positive", "negative"]
print("VADER checks passed.")


VADER checks passed.


In [46]:
started = time.perf_counter()
vader_train_records = vader_corpus(imdb)
assert len(vader_train_records) == len(imdb) == 25_000
assert all(record["row_id"] == i for i, record in enumerate(vader_train_records))
print(f"Scored {len(vader_train_records):,} reviews in {time.perf_counter() - started:.1f} s")
print("VADER labels:", Counter(record["label"] for record in vader_train_records))
print("Scores equal to zero:", sum(record["compound"] == 0 for record in vader_train_records))
for i in range(3):
    print(f"Review {i}: word-count={sentiment_records[i]['label']}, "
          f"VADER={vader_train_records[i]['label']}, "
          f"compound={vader_train_records[i]['compound']:.4f}")

Scored 25,000 reviews in 79.9 s
VADER labels: Counter({'positive': 16441, 'negative': 8559})
Scores equal to zero: 18
Review 0: word-count=negative, VADER=negative, compound=-0.1167
Review 1: word-count=negative, VADER=negative, compound=-0.8648
Review 2: word-count=positive, VADER=positive, compound=0.8710


## VADER score for the selected entity

Calculate the mean of the whole-review compound scores for the matched review IDs. Compare it with the earlier positive/negative count summary. Explain what the two scores do and do not measure.

In [47]:
# Reuse the entity matches found in exercise 2B.
if entity_results:
    entity_compounds = [vader_train_records[result["row_id"]]["compound"]
                        for result in entity_results]
    mean_compound = sum(entity_compounds) / len(entity_compounds)
    print(f"{ENTITY}: mean review-level compound score = {mean_compound:.4f}")
    print("Number of NER-confirmed reviews:", len(entity_compounds))
else:
    mean_compound = None
    print("No NER-confirmed reviews: no entity score can be calculated.")

Paris: mean review-level compound score = 0.6103
Number of NER-confirmed reviews: 237


# Guided exercise 4 — Evaluate both sentiment methods (10 minutes)

## Evaluation data

Download the **official IMDb test split** with `datasets.load_dataset`. This split is separate from the training reviews used in the earlier exercises. Its reference labels are 0 for negative and 1 for positive. `datasets` may already have cached the split when it loaded IMDb earlier; running `load_dataset` still explicitly selects the test data.

For a manageable classroom runtime, shuffle this test split with seed 42 and evaluate the **first 2,000 shuffled reviews**. Both methods use the same texts and reference labels in the same order. This is a reproducible sample of the test split, not a score on all 25,000 test reviews. Keep the threshold fixed before inspecting the test results. Repeated tuning on this sample would make it less suitable as a final unbiased evaluation.

The earlier counting rule and the VADER rule are lexicon-based: neither is trained on the IMDb training split in this notebook. A separate test split still allows a consistent comparison on reviews outside the working corpus.

To evaluate the full test split after class, set `N_EVALUATION = len(imdb_test_full)` in the next cell and rerun the evaluation. This takes longer.


In [48]:
imdb_test_full = load_dataset("stanfordnlp/imdb", split="test")
N_EVALUATION = 2_000
imdb_test = imdb_test_full.shuffle(seed=42).select(range(N_EVALUATION))
references = list(imdb_test["label"])
print("Full IMDb test split:", len(imdb_test_full))
print("Evaluation sample:", len(imdb_test))
print("Reference labels:", Counter(references), "(0 = negative, 1 = positive)")
assert len(imdb_test_full) == 25_000 and len(references) == N_EVALUATION

Full IMDb test split: 25000
Evaluation sample: 2000
Reference labels: Counter({1: 1000, 0: 1000}) (0 = negative, 1 = positive)


In [49]:
started = time.perf_counter()
# Predict each method from review text only; reference labels are used afterwards.
test_count_records = classify_corpus(imdb_test)
test_vader_records = vader_corpus(imdb_test)
label_id = {"negative": 0, "positive": 1}
count_predictions = [label_id[record["label"]] for record in test_count_records]
vader_predictions = [label_id[record["label"]] for record in test_vader_records]
assert len(count_predictions) == len(vader_predictions) == len(references)
print(f"Predictions for both methods: {time.perf_counter() - started:.1f} s")

Predictions for both methods: 9.1 s


## Accuracy, precision, recall and F1

The current Hugging Face `datasets` library loads the data; the companion **`evaluate`** library loads the metrics. All four measures range from 0 to 1. Here label 1 (positive review) is the positive class for precision, recall and F1.

- **Accuracy:** fraction of all reviews classified correctly.
- **Precision:** fraction of predicted positive reviews whose reference label is positive.
- **Recall:** fraction of reference-positive reviews predicted positive.
- **F1:** harmonic mean of precision and recall.

The metric modules are downloaded from the Hub the first time they are loaded. Compare the two outputs without assuming that one method must be better. Both use the same arbitrary treatment of ties / compound 0 as negative.


In [50]:
import evaluate

# Hugging Face Evaluate computes the selected classification metrics together.
metrics = evaluate.combine(["accuracy", "precision", "recall", "f1"])
count_scores = metrics.compute(predictions=count_predictions, references=references)
vader_scores = metrics.compute(predictions=vader_predictions, references=references)
print("Evaluation on", len(references), "IMDb test reviews")
print("Word-count lexicon:", {name: round(value, 4) for name, value in count_scores.items()})
print("VADER:", {name: round(value, 4) for name, value in vader_scores.items()})

Evaluation on 2000 IMDb test reviews
Word-count lexicon: {'accuracy': 0.7095, 'precision': 0.699, 'recall': 0.736, 'f1': 0.717}
VADER: {'accuracy': 0.6995, 'precision': 0.651, 'recall': 0.86, 'f1': 0.7411}


In [53]:
# Display two disagreements with the reference label for each method.
for method, predictions in (("word-count", count_predictions), ("VADER", vader_predictions)):
    errors = [i for i, (prediction, label) in enumerate(zip(predictions, references))
              if prediction != label]
    print(f"\n{method}: {len(errors)} errors")
    for i in errors[:2]:
        print("Reference:", references[i], "prediction:", predictions[i])
        print(clean_review(imdb_test[i]["text"])[:500])


word-count: 581 errors
Reference: 1 prediction: 0
This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French answer to James Bond). The series was launched in the early 1950's, and spawned at least eight films (none of which was ever released in the U.S.). 'O.S.S.117:Cairo,Nest Of Spies' is a breezy little comedy that should not...repeat NOT, be taken too seriously. Our protagonist finds himself in the middle of a spy chase in Egypt (with Morroco doing stand in for Egypt) to find out about a long lost fri
Reference: 1 prediction: 0
I was truly and wonderfully surprised at "O' Brother, Where Art Thou?" The video store was out of all the movies I was planning on renting, so then I came across this. I came home and as I watched I became engrossed and found myself laughing out loud. The Coen's have made a magnificiant film again. But I think the first time you watch this movie, you get to know the characters. The second time, now that you know them,

**Discussion:** Which method has higher accuracy on this sample? Does it also have higher precision and recall? Find a mistake involving negation or mixed sentiment if one appears. How might the results change with a different domain, another test sample or a neutral class? The tests evaluate overall review sentiment, not sentiment towards a named entity.

- **word-count seems to have higher accuracy, higher but lower recall than VADER. A comment "Peter [Hyams] has no talent as a director" seems to be an example where the negation makes a significant difference in the meaning behind the comment. The results may be more accurate in different domains/test samples where the content being reviewed could be less varied, less opinionated, or more neutral (like legal documentations).**

**Documentation:** [NLTK VADER](https://www.nltk.org/howto/sentiment.html), [IMDb dataset](https://huggingface.co/datasets/stanfordnlp/imdb), [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/a_quick_tour).


## What to submit

Submit this completed notebook with outputs saved, plus a short explanation in English covering:

1. The Jaccard, count-cosine and TF-IDF cosine matrices, their best distinct sentence pairs, and how repetitions and IDF weighting affect the measures.
2. The number of IMDb reviews classified, the predicted label distribution, ties, and one limitation of the lexicon rule.
3. Your entity query, positive and negative retrieved examples, the score and its denominator; if no match was found, report that fact and try another query.
4. Your VADER review scores, binary predictions, and mean score for the selected entity. Explain why review sentiment is not necessarily sentiment about the entity.
5. The sample size and evaluation metrics for both sentiment methods on the held-out IMDb test sample, plus one limitation.

**References**
- [NLTK book: named entity recognition](https://www.nltk.org/book/ch07.html)
- [NLTK opinion lexicon documentation](https://www.nltk.org/api/nltk.corpus.reader.opinion_lexicon.html)
- [IMDb Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/)
- [IMDb dataset card](https://huggingface.co/datasets/stanfordnlp/imdb)
